# 00. Setup del catálogo, el volume, las máscaras y los permisos

Corre **antes de desplegar el bundle**, una vez por entorno (`electrocasa_dev` y `electrocasa_prod`).

Requisitos previos, desde la interfaz del workspace:

1. Crear los grupos `electrocasa_ingenieria`, `electrocasa_analistas` y `electrocasa_auditoria` en **Settings > Identity and access > Groups** (no con la API ni desde un notebook: así quedarían como grupos locales y el `GRANT` fallaría con `PRINCIPAL_DOES_NOT_EXIST`).
2. Agregarse como miembro de `electrocasa_ingenieria`. El dueño del pipeline tiene que estar en ese grupo para que las tablas derivadas de Empleados guarden los valores reales.

Antes de subir el notebook al repositorio, limpiar las salidas (**Clear all outputs**) para no versionar datos de la sesión.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if not CATALOGO or not all(c.isalnum() or c == "_" for c in CATALOGO):
    raise ValueError("El catálogo solo admite letras, números y guion bajo")

In [ ]:
GRUPO_INGENIERIA = "electrocasa_ingenieria"
GRUPO_ANALISTAS = "electrocasa_analistas"
GRUPO_AUDITORIA = "electrocasa_auditoria"
FUENTES = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
RUTA_LANDING = f"/Volumes/{CATALOGO}/default/vol_landing"


def ejecutar(sentencia):
    print(sentencia.strip())
    return spark.sql(sentencia)

## 1. Catálogo, schemas por capa y volume de landing

En un workspace serverless (o en Free Edition) el catálogo usa el almacenamiento por defecto y no pide `MANAGED LOCATION`. Todas las tablas del proyecto son managed.

In [ ]:
ejecutar(f"CREATE CATALOG IF NOT EXISTS {CATALOGO} COMMENT 'Proyecto integrador ElectroCasa'")
for esquema in ("bronze", "silver", "gold"):
    ejecutar(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")
ejecutar(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.default.vol_landing COMMENT 'Landing de archivos por fuente'")
for fuente in FUENTES:
    dbutils.fs.mkdirs(f"{RUTA_LANDING}/{fuente}")
display(dbutils.fs.ls(RUTA_LANDING))

## 2. Funciones de máscara para `dni` y `salario`

Solo `electrocasa_ingenieria` ve el valor real. El resto ve el `dni` con sus tres últimos dígitos y el salario como nulo (la columna es numérica, así que no puede llevar asteriscos). Las tablas del pipeline declaran la máscara en su propio schema (cláusula `MASK`), por eso estas funciones tienen que existir antes del primer despliegue.

In [ ]:
ejecutar(f"""
CREATE FUNCTION IF NOT EXISTS {CATALOGO}.silver.mascara_dni(dni STRING)
RETURNS STRING
COMMENT 'Muestra el dni real solo a electrocasa_ingenieria'
RETURN CASE
  WHEN is_account_group_member('{GRUPO_INGENIERIA}') THEN dni
  WHEN dni IS NULL THEN NULL
  ELSE concat('*****', right(dni, 3))
END
""")
ejecutar(f"""
CREATE FUNCTION IF NOT EXISTS {CATALOGO}.silver.mascara_salario(salario DECIMAL(10,2))
RETURNS DECIMAL(10,2)
COMMENT 'Muestra el salario real solo a electrocasa_ingenieria'
RETURN CASE WHEN is_account_group_member('{GRUPO_INGENIERIA}') THEN salario ELSE NULL END
""")
display(spark.sql(f"SELECT is_account_group_member('{GRUPO_INGENIERIA}') AS soy_miembro_de_ingenieria"))

Si la última consulta devuelve `false` justo después de agregarse al grupo, esperar alrededor de un minuto y repetirla: `is_account_group_member` puede tardar en reconocer la membresía.

## 3. Permisos por grupo

| Grupo | Acceso |
|---|---|
| `electrocasa_ingenieria` | Lectura y escritura sobre bronze, silver y gold, y lectura y escritura del volume |
| `electrocasa_analistas` | Solo lectura sobre gold |
| `electrocasa_auditoria` | Solo lectura sobre gold y evidencia de auditoría y lineage |

In [ ]:
for grupo in (GRUPO_INGENIERIA, GRUPO_ANALISTAS, GRUPO_AUDITORIA):
    ejecutar(f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `{grupo}`")

for esquema in ("bronze", "silver", "gold"):
    ejecutar(f"GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW ON SCHEMA {CATALOGO}.{esquema} TO `{GRUPO_INGENIERIA}`")
ejecutar(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.default TO `{GRUPO_INGENIERIA}`")
ejecutar(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {CATALOGO}.default.vol_landing TO `{GRUPO_INGENIERIA}`")
ejecutar(f"GRANT EXECUTE ON FUNCTION {CATALOGO}.silver.mascara_dni TO `{GRUPO_INGENIERIA}`")
ejecutar(f"GRANT EXECUTE ON FUNCTION {CATALOGO}.silver.mascara_salario TO `{GRUPO_INGENIERIA}`")

for grupo in (GRUPO_ANALISTAS, GRUPO_AUDITORIA):
    ejecutar(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOGO}.gold TO `{grupo}`")

In [ ]:
for objeto in (f"CATALOG {CATALOGO}", f"SCHEMA {CATALOGO}.bronze", f"SCHEMA {CATALOGO}.silver", f"SCHEMA {CATALOGO}.gold"):
    print(objeto)
    display(spark.sql(f"SHOW GRANTS ON {objeto}"))

## 4. Demostración de `REVOKE`

Se otorga a `electrocasa_analistas` un privilegio que no le corresponde (`SELECT` sobre silver), se muestra el `SHOW GRANTS` antes y después de revocarlo. Las dos salidas van al README como evidencia.

In [ ]:
ejecutar(f"GRANT SELECT ON SCHEMA {CATALOGO}.silver TO `{GRUPO_ANALISTAS}`")
print("Antes del REVOKE")
display(spark.sql(f"SHOW GRANTS `{GRUPO_ANALISTAS}` ON SCHEMA {CATALOGO}.silver"))

In [ ]:
ejecutar(f"REVOKE SELECT ON SCHEMA {CATALOGO}.silver FROM `{GRUPO_ANALISTAS}`")
print("Después del REVOKE")
display(spark.sql(f"SHOW GRANTS `{GRUPO_ANALISTAS}` ON SCHEMA {CATALOGO}.silver"))

## 5. Acceso de Auditoría a la evidencia

Se intenta otorgar a `electrocasa_auditoria` el acceso a `system.access`. Si el workspace no lo permite (en Free Edition ni el administrador tiene `MANAGE` sobre el catálogo `system`), la celda imprime el error completo para llevarlo al README, y la evidencia alternativa es la pestaña **Lineage** de Catalog Explorer.

In [ ]:
SENTENCIAS_SISTEMA = [
    f"GRANT USE CATALOG ON CATALOG system TO `{GRUPO_AUDITORIA}`",
    f"GRANT USE SCHEMA ON SCHEMA system.access TO `{GRUPO_AUDITORIA}`",
    f"GRANT SELECT ON TABLE system.access.audit TO `{GRUPO_AUDITORIA}`",
    f"GRANT SELECT ON TABLE system.access.table_lineage TO `{GRUPO_AUDITORIA}`",
    f"GRANT SELECT ON TABLE system.access.column_lineage TO `{GRUPO_AUDITORIA}`",
]
for sentencia in SENTENCIAS_SISTEMA:
    try:
        ejecutar(sentencia)
        print("Otorgado")
    except Exception as error:
        print(f"No se pudo otorgar: {type(error).__name__}: {error}")